# Governance and Compliance Analysis

### Dataset contextualization

The dataset collects personal and financial information on 500+ credit applications. Some names can be anonymized, but individuals can still be identifiable indirectly.

The data available can be profiled in different ways:

* Demographic Data: full_name, email, ssn, ip_address, gender, zip_code, date_of_birth;
* Financial Data: annual_income, credit_history_months, debt_to_income, savings_balance, annual_salary;
* Decision Data: loan_approved, rejection_reason, loan_purpose, interest_rate, approved_amount;
* Spending behaviour data: category, amount;
* Operational data: processing_timestamp.

### GDPR contextualization

GDPR (General Data Protection Regulation) is an European Union law that lays down rules relating to the protection of natural persons with regard to the processing of personal data and rules relating to the free movement of personal data (Art. 1 GDPR).

There are important definitions to take into account before we use the regulations in the context of this dataset:

* Personal Data (Art. 4 GDPR (1)): any information relating to an identified or identifiable natural person (‘data subject’); an identifiable natural person is one who can be identified, directly or indirectly, in particular by reference to an identifier such as a name, an identification number, location data, an online identifier or to one or more factors specific to the physical, physiological, genetic, mental, economic, cultural or social identity of that natural person;
* Profiling Data (Art. 4 GDPR (4)): any form of automated processing of personal data consisting of the use of personal data to evaluate certain personal aspects relating to a natural person, in particular to analyse or predict aspects concerning that natural person’s performance at work, economic situation, health, personal preferences, interests, reliability, behaviour, location or movements.

According to the two definitions above, we can classify each data category as personal or profiling:

* Demographic Data: Personal Data
* Financial Data: Personal Data
* Decision Data: Profiling Data
* Spending Behaviour Data: Personal Data
* Operational Data: Personal Data



## Personal Identifiable Information (PII)

Important definitions: 

* Direct identifiers are unique to a person, a single direct identifier is typically enough to determine someone's identity. It's very-high risk information.
* Indirect identifiers are not unique, a single indirect identifier is not enough to identify a person, but a combination can. It's medium to high risk information.

We already classified what is considered personal data and what it's not, so now let's classify them as direct or indirect identifiers:

| Field                 | Data Category      | PII Classification   |
|:----------------------|:-------------------|:---------------------|
| ssn                   | Demographic        | Direct Identifier    |
| ip_address            | Demographic        | Direct Identifier    |
| email                 | Demographic        | Direct Identifier    |
| full_name             | Demographic        | Direct Identifier    |
| gender                | Demographic        | Indirect Identifier  |
| zip_code              | Demographic        | Indirect Identifier  |
| date_of_birth         | Demographic        | Indirect Identifier  |
| annual_income         | Financial          | Indirect Identifier  |
| credit_history_months | Financial          | Indirect Identifier  |
| debt_to_income        | Financial          | Indirect Identifier  |
| savings_balance       | Financial          | Indirect Identifier  |
| annual_salary         | Financial          | Indirect Identifier  |
| category              | Spending Behaviour | Indirect Identifier  |
| amount                | Spending Behaviour | Indirect Identifier  |
| processing_timestamp  | Operational        | Indirect Identifier  |

## Pseudonymization & Anonymization

Definitions: 

* Pseudonymization (Art. 4(5) GDPR): The processing of personal data in such a manner that the personal data can no longer be attributed to a specific data subject without the use of additional information. Action that can be reversible.
* Anonymization: Data processing technique that removes or modifies PII to ensure individuals cannot be identified directly or indirectly, while retaining utility. Action not reversible.

To show pseudonymization (if we did anonymization, we would use the same examples), we are going to use a direct identifier, like email. Since this is just a demonstration, we will work only with few random examples on the dataset.

### Pseudonymization

In [22]:
import pandas as pd
import hashlib

# Example dataset
df = pd.DataFrame({
    "email": [
        "jerry.smith17@hotmail.com",
        "brandon.walker2@yahoo.com",
        "scott.moore94@mail.com"
    ],
    "annual_income": [73000, 78000, 61000]
})

def pseudonymize(value):
    return hashlib.sha256(value.encode()).hexdigest()


df["email_pseudonymized"] = df["email"].apply(pseudonymize)

df_pseudonymized = df.drop(columns=["email"])

df_pseudonymized

,annual_income,email_pseudonymized
0,73000,116648a7761525746032d0ab323ceb01f50d11f7935164...
1,78000,c3522c0b54ef9045c73186bcabb53f8e512360ed17e9cc...
2,61000,b299e7d6a37e183bab209eb8df919652117dd16ed16698...


This is perceived as pseudonymization because the direct identifier (email) was removed, but we can still identify a person thanks to record linkage(same person connected to the same pseudonym). Re-Identification still possible with the use of the original email list.

### GDPR 7 Key Principles (Art. 5 GDPR)

 1) Lawfulness, Fairness & Transparency: Process data legally, openly and transparently;
 2) Purpose Limitation: Collect data only for specified and legitimate purposes;
 3) Data Minimization: Use data for adequate, relevant and limited purposes;
 4) Accuracy: Keep data accurate and updated;
 5) Storage Limitation: Retain data for no longer than necessary;
 6) Integrity & Confidentiality: Secure against unauthorized access;
 7) Accountability: Demonstrate compliance.

Following our data quality report:

* Uniqueness: Identified 2 duplicate records (0.40%) based on the _id field.
* Completeness: Missing values in ssn (1%), ip_address (1%), and annual_income (1%).
* Consistency: Mixed data types in annual_income (str,int,float) and inconsistent gender coding featuring six distinct variations (Male, M, F, Female, '', nan)
* Validity: Negative values in credit_history_months, which is logically impossible for lending.
* Accuracy: Inconsistent date formats in date_of_birth mixing ISO standard (YYYY-MM-DD) with European formats (DD/MM/YYYY) preventing correct age calculation.
* Governance Gaps: Heavy missingness in processing_timestamp (87.6%) and loan_purpose (90.0%)


| GDPR Principle                      | Dataset Observations                                                                                                         |
|:------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------|
| Lawfulness, Fairness & Transparency | Automated loan decisions with limited transparency due to missing processing_timestamp (87.6%) and loan_purpose (90.0%).     |
| Purpose Limitation                  | Dataset appears intended for credit risk assessment, but missing loan_purpose (90.0%) reduces clarity on decision context.   |
| Data Minimization                   | Granular spending behavior categories may exceed what is strictly necessary for lending decisions.                           |
| Accuracy                            | Inconsistent date_of_birth formats, mixed data types in annual_income, and invalid negative values in credit_history_months. |
| Storage Limitation                  | Lack of processing_timestamp (87.6%) for most records prevents assessment of data retention duration.                        |
| Integrity & Confidentiality         | Sensitive financial and behavioral data present; partial missingness in ssn and ip_address indicates inconsistent handling.  |
| Accountability                      | Duplicate records (0.40%) and major governance gaps suggest weak data controls and limited auditability.                     |

Other GDPR rules comprimised:

* Right to information (Art. 13,14 GDPR): Data subjects should have knowledge regarding what information is being processed and the period for which the personal data will be stored, but since processing_timestamp is missing in the majority of cases, that information cannot be provided;
* Right to rectification (Art. 16 GDPR): Individuals must be able to correct inaccurate data, but unable to do that because the data is inconsistent and also invalid in some fields;
* Automated individual decision-making (Art. 22 GDPR): Safeguards required for automated decisions with legal effects. The algorithimic loan approvals are done with low auditability.

## GDPR Compliance Analysis

Compliance analysis ensures an organization adheres to legal, regulatory, and industry standards, protecting it from lawsuits and reputational damage. 

With GDPR mapping we understood what laws and rights the organization was comprimising because of data quality issues, now we will understand real impact on individuals and institutional risk. 

Below is a summary table for our compliance analysis:



| GDPR Regulation                      | Compliance Impact                                                                                              
|:-------------------------------------|:---------------------------------------------------------------------------------------------------------------
| Lawfulness, Fairness & Transparency  | Individuals would not be able to understand when their data was processed or why a specific decision was made.
| Purpose Limitation                   | Weak documentation of processing purpose undermines lawful processing.                                        
| Data Minimization                    | Excessive data increases privacy risk.                                                                         
| Accuracy                             | Inaccurate data can lead to unfair financial decisions.                                                        
| Storage Limitation                   | Individuals unable to request deletion of data after it is no longer needed.                                
| Integrity & Confidentiality          | Inconsistent data handling suggests poor security practices.                                                   
| Accountability                       | Lack of accountability measures raises concerns about responsible data management.                             
                    

### EU AI Act Classification of Credit Scoring Systems

The EU AI act adresses the risks of AI to foster trustworthy AI in Europe.
The AI Act defines 4 levels of risk for AI systems:

* Unacceptable Risk: All AI systems considered a clear threat to the safety, livelihoods and rights of people are banned;
* High Risk: AI use cases that can pose serious risks to health, safety or fundamental rights;
* Transparency/Limited Risk: This refers to the risks associated with a need for transparency around the use of AI;
* Minimal or No Risk: The AI Act does not introduce rules for AI that is deemed minimal or no risk. This includes applications such as AI-enabled video games or spam filters.

According to EU AI Act Annex III section 5, credit score systems are High-risk AI systems as they are used to "give access to essential private and public services" and make decisions with significant legal and economic effects for individuals. 

High-risk AI systems are subject to strict obligations before entering the market, let's see if NovaCred complies with those obligations:



| Obligations                | Dataset Issue                                                                                                                     |
|:---------------------------|:----------------------------------------------------------------------------------------------------------------------------------|
| High Data Quality          | Duplicate records (0.40%), missing values in ssn/ip_address/annual_income (1%), inconsistent data types in annual_income and more |
| Traceability & Logging     | Heavy missingness in processing_timestamp (87.6%) prevents audit trails and traceability of automated decisions                   |
| Bias Mitigation            | Inconsistent gender coding and missing values may introduce bias in credit scoring models                                         |
| Human Oversight            | Low auditability of loan approvals and missing loan_purpose (90.0%) makes human review and override difficult                     |
| Robustness & Cybersecurity | Inconsistent data handling and partial missingness in sensitive fields (ssn, ip_address) suggests poor security practices         |
| Documentation              | Missing loan_purpose (90.0%) and processing_timestamp (87.6%) prevent clear documentation of decision logic                       |

## Policy Recommendations

To ensure GDPR and EU AI Act compliance , NovaCred should make effort by implementing policies for different purposes, for example, like:

* Data Quality & Validation Policy: All personal and financial data used for credit decision-making purposes should mandatorily pass validation checks before processing.
    * Measures: 
        * Enforces consistent data types for financial data (annual_income, for example);
        * Prevents duplicate records;
        * Enforces value restrictions for certain fields (like credit_history_months);
        * Standardize formats for dates, avoiding inconsistencies.

* Data Minimization Policy: Only data strictly necessary to make credit decisions may be used. Periodic reviews should be made.
    * Measures:
        * Prohibits the use of highly granular spending behaviour categories;
        * Prohibits the use of unnecessary personal data, like gender.

* Transparency & Explainability Policy: Individuals must receive meaningful information about automated credit decisions.
    * Measures:
        * Documentation about decision logic at high detailed level;
        * Provide clear explanations for loan rejections;
        * Record and retain processing_timestamp for every decision.

* Data Retention Policy: Retain personal data only for defined and documented periods.
    * Measures:
        * Define retention periods for every data category;
        * Delete automatically records after retention period expiry;
        * All processed events should have a timestamp attached.

* Accountability Policy: The organization should always be able to clearly demonstrate GDPR compliance.
    * Measures:
        * Document all governance decisions and policy exceptions;
        * Perform periodic internal audits;
        * Assign clear roles in data ownership and data stewardship to ensure high-quality and secure data.

* Automated Decision-Making Safeguards Policy: All automated credit decisions must include human oversight and feedback mechanisms.
    * Measures:
        * Enable manual override with meaningful justification;
        * Credit decisions with high-impact require mandatory human review.

* Information Access Control Policy: Access to personal and financial data should be as restricted as possible.
    * Measures:
        * Apply data encryption in areas where the information is not required;
        * Restrict access to sensitives attributes, like SSN and IP address.
